# 10 — The YAML Rule DSL

COGANT's translation layer turns graph nodes into Active Inference roles. Most of the time you don't need Python to extend it — the **YAML rule DSL** lets you declare new rules as data. Notebook 05 showed a first pass; this notebook is the reference card: the full schema, a multi-rule example, and a per-rule firing report.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/10_rule_dsl.ipynb
```

## 1. The DSL schema

A YAML rule file has a single top-level key `rules`, whose value is a list of rule declarations. Each rule has:

| Field | Type | Purpose |
| --- | --- | --- |
| `name` | string | Human-readable identifier (unique per file). |
| `role` | string | Active Inference role the rule assigns on match. |
| `confidence` | float | Score in `(0, 1]` — higher wins in conflicts. |
| `description` | string | Optional free-text docs. |
| `conditions` | list | One or more conditions; **all** must match (AND). |

Supported condition keys are `node_kind`, `name_pattern` (a glob), `has_method`, and `edge_type`.

In [ ]:
from __future__ import annotations

RULES_YAML = '''
rules:
  - name: HandlerMethodsAreActions
    role: ACTION
    confidence: 0.88
    description: Methods ending in _handler perform a side effect.
    conditions:
      - node_kind: method
        name_pattern: '*_handler'

  - name: SensorClassesAreObservations
    role: OBSERVATION
    confidence: 0.90
    description: Classes named Sensor* read from the environment.
    conditions:
      - node_kind: class
        name_pattern: 'Sensor*'

  - name: ValidatorFunctionsAreConstraints
    role: CONSTRAINT
    confidence: 0.80
    description: Module-level validate_* functions enforce invariants.
    conditions:
      - node_kind: function
        name_pattern: 'validate_*'

  - name: PlannerClassesArePolicies
    role: POLICY
    confidence: 0.75
    description: Classes ending in Planner select actions.
    conditions:
      - node_kind: class
        name_pattern: '*Planner'
'''
print(RULES_YAML.strip())

## 2. Load and compile

The loader turns a Python dict (or a YAML file path) into a `DSLRuleSet`. The compiler turns each declarative rule into a `CompiledRule` with a `match(node, graph) -> float` method.

In [ ]:
import yaml

from cogant.translate.dsl import compile_ruleset, load_rules_from_dict

ruleset = load_rules_from_dict(yaml.safe_load(RULES_YAML))
compiled = compile_ruleset(ruleset)

print(f"Loaded   : {len(ruleset.rules)} rules")
print(f"Compiled : {len(compiled)} CompiledRule objects")
for r in ruleset.rules:
    print(f"  - {r.name:<32} role={r.role:<10} conf={r.confidence}")

## 3. Build a small graph that exercises every rule

We hand-construct a `ProgramGraph` with five nodes: one match per rule plus one that should *not* match any rule. Hand-building lets us reason about ground truth without having to scan a repository.

In [ ]:
from cogant.schemas.core import Edge, EdgeKind, Node, NodeKind
from cogant.schemas.graph import GraphMetadata, ProgramGraph

graph = ProgramGraph(metadata=GraphMetadata(repo_uri="synthetic://rule-dsl-demo"))

specs = [
    ("n_click", NodeKind.METHOD,   "click_handler",    "demo.Ui.click_handler"),
    ("n_temp",  NodeKind.CLASS,    "SensorTemperature","demo.SensorTemperature"),
    ("n_val",   NodeKind.FUNCTION, "validate_config",  "demo.validate_config"),
    ("n_plan",  NodeKind.CLASS,    "RoutePlanner",     "demo.RoutePlanner"),
    ("n_util",  NodeKind.FUNCTION, "helper",           "demo.helper"),
]
for nid, kind, name, qname in specs:
    graph.add_node(Node(id=nid, kind=kind, name=name, qualified_name=qname))

print(f"Graph: {len(graph.nodes)} nodes")

## 4. Apply the rules and record every firing

For every `(node, rule)` pair we compute a match score. Non-zero scores mean the rule *fired* on that node. Zero means no match. We record firings as a small table.

In [ ]:
firings: list[dict] = []
for node in graph.nodes.values():
    for rule in compiled:
        score = rule.match(node, graph)
        if score > 0.0:
            firings.append({
                "node": node.name,
                "kind": node.kind.value,
                "rule": rule.name,
                "role": rule.role,
                "score": round(score, 3),
            })

print(f"Total firings: {len(firings)}")
print()
header = f"{'node':<20} {'kind':<10} {'rule':<35} {'role':<12} {'score':>6}"
print(header)
print("-" * len(header))
for f in firings:
    print(f"{f['node']:<20} {f['kind']:<10} {f['rule']:<35} {f['role']:<12} {f['score']:>6.3f}")

## 5. Per-node winning role

When multiple rules match a single node the highest-confidence match wins. Even though our example has no overlaps, the same collapse step is what a real pipeline does before handing the mapping off to GNN emission.

In [ ]:
winners: dict[str, dict] = {}
for f in firings:
    current = winners.get(f["node"])
    if current is None or f["score"] > current["score"]:
        winners[f["node"]] = f

print("Winning role per node:")
for node_name in sorted(n.name for n in graph.nodes.values()):
    if node_name in winners:
        w = winners[node_name]
        print(f"  {node_name:<20} -> {w['role']:<12} (rule={w['rule']}, score={w['score']:.2f})")
    else:
        print(f"  {node_name:<20} -> (no match)")

## 6. Assert the expected match set

Exactly four nodes should have matched — one per rule — and `helper` should have no match.

In [ ]:
expected = {
    "click_handler": "ACTION",
    "SensorTemperature": "OBSERVATION",
    "validate_config": "CONSTRAINT",
    "RoutePlanner": "POLICY",
}
actual = {name: w["role"] for name, w in winners.items()}

assert actual == expected, f"role mismatch: {actual} vs {expected}"
assert "helper" not in winners, "helper should not match any rule"
print("OK — every rule fired exactly once, on the expected node.")

## 7. Where to go next

- **File-based rules:** save the YAML to `custom_rules.yaml` and load it with `load_rules_from_yaml(path)`.
- **Combine with the built-ins:** the translation engine applies your custom rules alongside ~18 built-ins. Keep confidences below `1.0` to let built-ins override yours when they should.
- **Dataflow conditions:** use `edge_type` to capture patterns like 'a method that has an outgoing WRITES edge', which is much stronger than a name glob.
- **Plugin escape hatch:** if the DSL isn't expressive enough, write a `RulePlugin` — notebook 09 shows the plugin skeleton.

## 8. Takeaways

- The DSL is YAML in, semantic mapping out — no Python required for most role adjustments.
- Conditions are ANDed within a rule; rules are ORed across the ruleset.
- Higher confidence wins in conflicts, so you can layer specific rules on top of general ones without mutual exclusion logic.